In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler, PowerTransformer
from sklearn.feature_selection import RFE
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Setup NLTK
nltk.data.path.append("/usr/share/nltk_data")

Task 1: Handling Missing Data – Titanic Dataset
Overview: Identify and resolve null values within the Titanic dataset.

Technique: Median Imputation for numerical values and dropping highly sparse columns.

Why: Median is robust to the skewness of 'Age'. Features like 'Cabin' are dropped as they lack enough data to be meaningful.

Results & Observations: All null values were removed; the dataset is now structurally complete for modeling

In [2]:
titanic = pd.read_csv('/kaggle/input/datasets/heptapod/titanic/train_and_test2.csv')

# Numerical Imputation and dropping redundant columns
titanic.fillna(titanic.median(numeric_only=True), inplace=True)
titanic.dropna(axis=1, thresh=len(titanic)*0.5, inplace=True) # Drop columns with >50% missing

print("Task 1 - Missing Values after cleaning:", titanic.isnull().sum().sum())

Task 1 - Missing Values after cleaning: 0


Task 2: Feature Encoding – Car Evaluation Dataset
Overview: Transform categorical car attributes into numerical formats.

Technique: One-Hot Encoding (OHE).

Why: The dataset consists entirely of categorical strings; OHE allows the model to treat categories as distinct features without implying a mathematical rank.

Results & Observations: The original 6 features expanded into 21 binary features.

In [3]:
car_df = pd.read_csv('/kaggle/input/datasets/meheralimeer/car-evaluation-dataset/car.data', 
                     names=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety', 'class'])

car_encoded = pd.get_dummies(car_df, columns=['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety'])
print(f"Task 2 - Encoded Shape: {car_encoded.shape}")

Task 2 - Encoded Shape: (1728, 22)


Task 3: Feature Scaling – Wine Quality Dataset
Overview: Adjust the scale of chemical attributes in wine data.

Technique: Min-Max Normalization.

Why: Normalization is used to bring all features into a [0, 1] range, preventing high-magnitude features (like 'Total Sulfur Dioxide') from dominating the model.

Results & Observations: All features now range from 0 to 1, ensuring uniform feature importance.

In [4]:
# --- Task 3: Feature Scaling – Wine Quality ---
try:
    # Use sep=None to let pandas auto-detect if it's a comma or semicolon
    wine = pd.read_csv('/kaggle/input/datasets/meheralimeer/wine-quality-dataset/winequality-red.csv', sep=None, engine='python')
    
    # Clean column names (removes invisible spaces/newlines)
    wine.columns = wine.columns.str.strip()
    
    # Check for 'quality' case-insensitively and drop it
    target_col = [col for col in wine.columns if col.lower() == 'quality']
    
    if target_col:
        col_to_drop = target_col[0]
        scaler = MinMaxScaler()
        # Scale everything EXCEPT the target column
        wine_scaled = pd.DataFrame(
            scaler.fit_transform(wine.drop(columns=[col_to_drop])), 
            columns=[c for c in wine.columns if c != col_to_drop]
        )
        print(f"Task 3 - Successfully scaled {wine_scaled.shape[1]} features.")
        print(f"Task 3 - Scale Range: [{wine_scaled.min().min()}, {wine_scaled.max().max()}]")
    else:
        print(f"Task 3 Warning: 'quality' column not found. Available: {list(wine.columns[:3])}...")

except Exception as e: 
    print(f"Task 3 Error: {e}")

Task 3 - Successfully scaled 11 features.
Task 3 - Scale Range: [0.0, 1.0000000000000002]


Task 4: Handling Outliers – Boston Housing Dataset
Overview: Detect and treat outliers in residential property values.

Technique: Interquartile Range (IQR) Method.

Why: IQR effectively captures the central 50% of the data and trims the extreme tails (noise).

Results & Observations: Outliers were removed, reducing the row count but improving data consistency.

In [5]:
boston = pd.read_csv('/kaggle/input/datasets/schirmerchad/bostonhoustingmlnd/housing.csv')

Q1 = boston.quantile(0.25)
Q3 = boston.quantile(0.75)
IQR = Q3 - Q1
boston_clean = boston[~((boston < (Q1 - 1.5 * IQR)) | (boston > (Q3 + 1.5 * IQR))).any(axis=1)]

print(f"Task 4 - Rows: {len(boston)} (Original) -> {len(boston_clean)} (Cleaned)")

Task 4 - Rows: 489 (Original) -> 444 (Cleaned)


Task 5: Data Imputation (Advanced) – Retail Sales Dataset
Overview: Use neighborhood-based logic to fill missing sales data.

Technique: KNN Imputation (k=3).

Why: KNN is more accurate than mean imputation because it predicts missing values based on similar data points.

Results & Observations: Missing numerical data was imputed based on the similarity of transaction patterns.

In [6]:
retail = pd.read_csv('/kaggle/input/datasets/mohammadtalib786/retail-sales-dataset/retail_sales_dataset.csv')
num_cols = retail.select_dtypes(include=[np.number]).columns

imputer = KNNImputer(n_neighbors=3)
retail[num_cols] = imputer.fit_transform(retail[num_cols])

print("Task 5 - KNN Imputation completed.")

Task 5 - KNN Imputation completed.


Task 6: Feature Engineering – Heart Disease Dataset
Overview: Derive new insights from existing clinical health data.

Technique: Interaction Feature Creation (Cholesterol to Age ratio).

Why: Cholesterol levels relative to age can provide a more nuanced risk signal than raw cholesterol alone.

Results & Observations: A new 'risk_ratio' feature was successfully synthesized.

In [7]:
try:
    # Defining standard UCI names since .data files often lack them
    heart_cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
    # Using the Cleveland processed data from your folder structure
    heart = pd.read_csv('/kaggle/input/datasets/moulishwaranbalaji/heart-disease-data/processed.cleveland.data', names=heart_cols, na_values="?")
    heart.fillna(heart.median(), inplace=True)
    heart['risk_ratio'] = heart['chol'] / (heart['age'] + 1)
    print("Task 6 Complete. Sample Risk Ratio:\n", heart[['age', 'chol', 'risk_ratio']].head(2))
except Exception as e: print(f"Task 6 Error: {e}")

Task 6 Complete. Sample Risk Ratio:
     age   chol  risk_ratio
0  63.0  233.0    3.640625
1  67.0  286.0    4.205882


Task 7: Variable Transformation – Bike Sharing Dataset
Overview: Correct the distribution of bike demand data.

Technique: Log Transformation.

Why: Demand (count) data is often skewed; a log transform compresses the range and makes the distribution more Gaussian.

Results & Observations: Skewness was significantly reduced after transformation.

In [8]:
bike = pd.read_csv('/kaggle/input/datasets/marklvl/bike-sharing-dataset/hour.csv')
bike['cnt_log'] = np.log1p(bike['cnt'])

print(f"Task 7 - Skew: {bike['cnt'].skew():.2f} (Orig) vs {bike['cnt_log'].skew():.2f} (Log)")

Task 7 - Skew: 1.28 (Orig) vs -0.82 (Log)


Task 8: Feature Selection – Diabetes Dataset
Overview: Determine the most significant health indicators for diabetes.

Technique: Recursive Feature Elimination (RFE) with a Random Forest base.

Why: RFE iteratively evaluates the model and removes the least important features, reducing noise.

Results & Observations: Top 5 features were selected, streamlining the dataset for performance.

In [9]:
diabetes = pd.read_csv('/kaggle/input/datasets/organizations/uciml/pima-indians-diabetes-database/diabetes.csv')
X, y = diabetes.drop('Outcome', axis=1), diabetes['Outcome']

rfe = RFE(estimator=RandomForestClassifier(n_estimators=10), n_features_to_select=5)
rfe.fit(X, y)

print("Task 8 - Selected Features:", X.columns[rfe.support_].tolist())

Task 8 - Selected Features: ['Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']


Task 9: Handling Imbalanced Data – Credit Card Fraud
Overview: Resolve the extreme imbalance between fraud and non-fraud cases.

Technique: SMOTE (Synthetic Minority Over-sampling Technique).

Why: SMOTE creates synthetic fraud cases to help the model learn the minority class effectively.

Results & Observations: Class distribution is now balanced at a 1:1 ratio.

In [10]:
fraud = pd.read_csv('/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv').sample(10000)
X_f, y_f = fraud.drop('Class', axis=1), fraud['Class']

X_res, y_res = SMOTE().fit_resample(X_f, y_f)
print(f"Task 9 - Class distribution: {np.bincount(y_res)}")

Task 9 - Class distribution: [9977 9977]


Task 10: Combining Multiple Datasets – MovieLens Dataset
Overview: Merge disparate movie files into a single analytical view.

Technique: Inner Joining on 'movie_id'.

Why: Unifying ratings with titles allows for interpretable recommendation analysis.

Results & Observations: Ratings and titles are now successfully combined.

In [11]:
r_path = '/kaggle/input/datasets/pengtan/idlhw4/ml-100k/u.data'
m_path = '/kaggle/input/datasets/pengtan/idlhw4/ml-100k/u.item'

ratings = pd.read_csv(r_path, sep='\t', names=['user_id', 'movie_id', 'rating', 'ts'])
movies = pd.read_csv(m_path, sep='|', names=['movie_id', 'title'], usecols=[0, 1], encoding='latin-1')

movie_merged = pd.merge(ratings, movies, on='movie_id')
print(f"Task 10 - Merged rows: {len(movie_merged)}")

Task 10 - Merged rows: 100000


Task 11: Dimensionality Reduction – MNIST Dataset
Overview: Reduce the 784-pixel feature space of hand-written digits.

Technique: Principal Component Analysis (PCA).

Why: High-dimensional data is slow to train; PCA compresses this into the most significant "components".

Results & Observations: Dimensions reduced to 15 while retaining primary variance.

In [12]:
from sklearn.datasets import load_digits
digits = load_digits()

pca = PCA(n_components=15)
X_pca = pca.fit_transform(digits.data)

print(f"Task 11 - Dimensions: {digits.data.shape[1]} -> {X_pca.shape[1]}")

Task 11 - Dimensions: 64 -> 15


Task 12: Text Preprocessing – IMDB Movie Reviews
Overview: Clean raw movie reviews for machine learning.

Technique: Regex cleaning, Stopword removal, and Stemming.

Why: Standardizes text and reduces the vocabulary by keeping only word roots.

Results & Observations: Noise was removed; text is now ready for vectorization.

In [13]:
imdb = pd.read_csv('/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv', nrows=5)
ps = PorterStemmer()

def clean_text(text):
    text = re.sub('[^a-zA-Z]', ' ', text.lower())
    return " ".join([ps.stem(w) for w in text.split() if w not in set(stopwords.words('english'))])

imdb['cleaned_review'] = imdb['review'].apply(clean_text)
print("Task 12 - Cleaned sample:", imdb['cleaned_review'].iloc[0][:100])

Task 12 - Cleaned sample: one review mention watch oz episod hook right exactli happen br br first thing struck oz brutal unfl


Task 13: Time-Series Preprocessing – Air Quality Dataset
Overview: Handle gaps and resampling in chronological air quality data.

Technique: Forward Fill (ffill).

Why: Air sensors often fail momentarily; forward fill propagates the last known value to maintain continuity.

Results & Observations: Temporal continuity was restored.

In [14]:
air = pd.read_csv('/kaggle/input/datasets/jasmeet0516/air-quality/Air_Quality.csv', sep=';')
air.ffill(inplace=True)

print("Task 13 - Continuity established via forward filling.")

Task 13 - Continuity established via forward filling.
